# FraudeCero · Grupo 6 · 00 · Ingesta (capa BRONCE)
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** descargar las dos fuentes oficiales del caso, registrarlas (URL, fecha, tamaño, huella MD5) y persistirlas **sin modificar** como tablas Delta de la capa **bronce**, con validaciones de conteo, esquema y nulos.

| # | Fuente | Tipo | URL exacta | Actualización |
|---|---|---|---|---|
| F1 | Credit Card Fraud Detection (ULB / Worldline) | CSV 150,8 MB · 284.807 transacciones | https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud | Estática (sept. 2013, 2 días) |
| F2 | Tarjetas de crédito y débito – Superintendencia Financiera de Colombia | API SODA (CSV) de datos.gov.co | https://www.datos.gov.co/resource/h2jg-r3zg.csv | Mensual (por entidad, franquicia e indicador) |

Ambas fuentes están autorizadas para el caso G6 en la sección 11 del protocolo (Kaggle y Superintendencia Financiera).

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "00_ingesta"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

## 1. Descarga de la fuente F1 (Kaggle)
Se descarga con `kagglehub` directamente desde Kaggle. **No se sube el archivo desde el navegador**: en una primera prueba la subida a Colab quedó cortada (172.441 filas en vez de 284.807) y eso alteró todos los resultados (ver *registro de desafíos*). Por eso se valida tamaño, número de líneas y MD5.

In [ ]:
import kagglehub
t0 = time.time()
URL_F1 = "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud"
os.makedirs(f"{LAKE}/landing", exist_ok=True)
carpeta = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
RUTA_F1 = f"{LAKE}/landing/creditcard.csv"
shutil.copy(os.path.join(carpeta, "creditcard.csv"), RUTA_F1)

def huella(ruta_archivo):
    md5 = hashlib.md5()
    lineas = 0
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            md5.update(bloque)
            lineas += bloque.count(b"\n")
    return md5.hexdigest(), lineas

md5_f1, lineas_f1 = huella(RUTA_F1)
meta_f1 = {"fuente": "F1_kaggle_creditcard", "url": URL_F1,
           "fecha_descarga": datetime.datetime.now().isoformat(timespec="seconds"),
           "archivo": "creditcard.csv", "bytes": os.path.getsize(RUTA_F1),
           "lineas": lineas_f1, "md5": md5_f1, "formato": "CSV"}
print(json.dumps(meta_f1, indent=2, ensure_ascii=False))
assert meta_f1["bytes"] == 150_828_752, "⚠️ El tamaño no coincide con el archivo oficial: descarga incompleta"
registrar("descarga_F1", t0, filas=lineas_f1 - 1)

## 2. Descarga de la fuente F2 (Superintendencia Financiera – datos.gov.co)
Se consume la API SODA del portal de Datos Abiertos Colombia. El parámetro `$limit` evita el corte por defecto de 1.000 filas.

In [ ]:
import requests
t0 = time.time()
URL_F2 = "https://www.datos.gov.co/resource/h2jg-r3zg.csv"
RUTA_F2 = f"{LAKE}/landing/sfc_tarjetas_credito_debito.csv"
resp = requests.get(URL_F2, params={"$limit": 5_000_000}, timeout=600)
resp.raise_for_status()
with open(RUTA_F2, "wb") as f:
    f.write(resp.content)

md5_f2, lineas_f2 = huella(RUTA_F2)
meta_f2 = {"fuente": "F2_sfc_tarjetas", "url": URL_F2 + "?$limit=5000000",
           "fecha_descarga": datetime.datetime.now().isoformat(timespec="seconds"),
           "archivo": "sfc_tarjetas_credito_debito.csv", "bytes": os.path.getsize(RUTA_F2),
           "lineas": lineas_f2, "md5": md5_f2, "formato": "CSV vía API REST (SODA)"}
print(json.dumps(meta_f2, indent=2, ensure_ascii=False))

with open(f"{EVID}/tablas/registro_fuentes.json", "w", encoding="utf-8") as f:
    json.dump([meta_f1, meta_f2], f, indent=2, ensure_ascii=False)
registrar("descarga_F2", t0, filas=lineas_f2 - 1)

## 3. Lectura y persistencia en BRONCE
- **F1** se lee con **esquema explícito** (31 columnas numéricas): es más rápido que `inferSchema` y detecta filas corruptas.
- **F2** se guarda **toda como texto**: la capa bronce conserva el dato tal como llegó; la tipificación ocurre en plata.
- A ambas se les agregan columnas de linaje: `_fuente`, `_archivo`, `_fecha_ingesta`.

In [ ]:
t0 = time.time()
esquema_f1 = T.StructType([T.StructField("Time", T.DoubleType())] +
                          [T.StructField(f"V{i}", T.DoubleType()) for i in range(1, 29)] +
                          [T.StructField("Amount", T.DoubleType()), T.StructField("Class", T.IntegerType())])

bronce_cc = (spark.read.option("header", True).option("mode", "PERMISSIVE").schema(esquema_f1).csv(RUTA_F1)
             .withColumn("_fuente", F.lit("kaggle_mlg-ulb_creditcardfraud"))
             .withColumn("_archivo", F.lit("creditcard.csv"))
             .withColumn("_fecha_ingesta", F.current_timestamp()))
bronce_cc = guardar(bronce_cc, "bronce", "transacciones")
registrar("bronce_transacciones", t0, df=bronce_cc)

t0 = time.time()
bronce_sfc = (spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(RUTA_F2)
              .withColumn("_fuente", F.lit("datos.gov.co_h2jg-r3zg"))
              .withColumn("_archivo", F.lit("sfc_tarjetas_credito_debito.csv"))
              .withColumn("_fecha_ingesta", F.current_timestamp()))
bronce_sfc = guardar(bronce_sfc, "bronce", "sfc_tarjetas")
registrar("bronce_sfc_tarjetas", t0, df=bronce_sfc)

## 4. Validación 1 · Conteo de filas
El conteo de F1 debe ser exactamente **284.807** (filas del archivo oficial). Si no coincide, el notebook se detiene.

In [ ]:
n_cc, n_sfc = bronce_cc.count(), bronce_sfc.count()
print(f"bronce.transacciones : {n_cc:,} filas · {len(bronce_cc.columns)} columnas")
print(f"bronce.sfc_tarjetas  : {n_sfc:,} filas · {len(bronce_sfc.columns)} columnas")
assert n_cc == 284_807, f"Conteo inesperado en F1: {n_cc}"
assert n_sfc > 0, "F2 llegó vacía"
print("✅ Conteos validados")

## 5. Validación 2 · Esquema
Se verifica que F1 tenga las 31 columnas esperadas con su tipo y que F2 traiga las 11 columnas del portal.

In [ ]:
bronce_cc.printSchema()
bronce_sfc.printSchema()
esperadas_sfc = {"tipoentidad", "codigoentidad", "nombreentidad", "fechacorte", "cod_uca", "nombre_uca",
                 "subcuenta", "descripcion", "persona_natural", "persona_juridica", "total_tarjetas"}
faltan = esperadas_sfc - set(bronce_sfc.columns)
assert not faltan, f"Faltan columnas en F2: {faltan}"
assert [c for c in bronce_cc.columns if not c.startswith("_")] == [f.name for f in esquema_f1.fields]
print("✅ Esquemas validados")

## 6. Validación 3 · Nulos por columna

In [ ]:
def nulos_por_columna(df):
    total = df.count()
    fila = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    return (pd.DataFrame({"columna": list(fila.keys()), "nulos": list(fila.values())})
              .assign(pct_nulos=lambda d: (d["nulos"] / total * 100).round(4)))

nulos_cc = nulos_por_columna(bronce_cc)
nulos_sfc = nulos_por_columna(bronce_sfc)
mostrar(nulos_cc[nulos_cc["nulos"] > 0] if (nulos_cc["nulos"] > 0).any() else nulos_cc.head(5))
mostrar(nulos_sfc)
print("Columnas de F1 con nulos:", int((nulos_cc["nulos"] > 0).sum()), "| Columnas de F2 con nulos:", int((nulos_sfc["nulos"] > 0).sum()))

## 7. Perfilado inicial (A1.5)
Tabla de perfilado con tipo de dato, % de nulos, valores únicos, mínimo, máximo y una observación de calidad. Se perfilan **12 columnas** (el mínimo exigido es 8).

In [ ]:
t0 = time.time()
obs = {
    "Time": "Segundos desde la 1.ª transacción; cubre ~48 h (2 días). Sirve para derivar la hora.",
    "V1": "Componente PCA anonimizado; media ≈ 0 por construcción.",
    "V4": "Componente PCA; de las más asociadas al fraude.",
    "V10": "Componente PCA; valores muy negativos en fraudes.",
    "V12": "Componente PCA; valores muy negativos en fraudes.",
    "V14": "Componente PCA; la variable más discriminante del fraude.",
    "V17": "Componente PCA; valores negativos en fraudes.",
    "Amount": "Monto en EUR, muy sesgado a la derecha; incluye montos 0.",
    "Class": "Etiqueta 1 = fraude; desbalance extremo (≈0,17 %).",
}
filas = []
total = bronce_cc.count()
for c, o in obs.items():
    r = bronce_cc.agg(F.countDistinct(c).alias("u"), F.min(c).alias("mn"), F.max(c).alias("mx"),
                      F.sum(F.col(c).isNull().cast("int")).alias("nn")).collect()[0]
    filas.append({"fuente": "F1", "columna": c, "tipo": dict(bronce_cc.dtypes)[c],
                  "pct_nulos": round(r["nn"] / total * 100, 4), "unicos": r["u"],
                  "minimo": round(r["mn"], 4), "maximo": round(r["mx"], 4), "observacion": o})

total_sfc = bronce_sfc.count()
obs_sfc = {
    "nombreentidad": "Nombre de la entidad emisora; requiere estandarizar espacios y mayúsculas.",
    "fechacorte": "Fecha de corte mensual (texto en bronce); se tipifica a fecha en plata.",
    "descripcion": "Indicador reportado (tarjetas vigentes, transacciones, etc.).",
    "total_tarjetas": "Valor del indicador (texto en bronce); se tipifica a número en plata.",
}
for c, o in obs_sfc.items():
    r = bronce_sfc.agg(F.countDistinct(c).alias("u"), F.min(c).alias("mn"), F.max(c).alias("mx"),
                       F.sum(F.col(c).isNull().cast("int")).alias("nn")).collect()[0]
    filas.append({"fuente": "F2", "columna": c, "tipo": "string (bronce)",
                  "pct_nulos": round(r["nn"] / total_sfc * 100, 4), "unicos": r["u"],
                  "minimo": str(r["mn"])[:40], "maximo": str(r["mx"])[:40], "observacion": o})

perfil = pd.DataFrame(filas)
perfil.to_csv(f"{EVID}/tablas/perfilado_bronce.csv", index=False)
mostrar(perfil)
registrar("perfilado", t0)

## 8. Las 5 V con evidencia numérica (A1.2)
Se calculan con los datos que acabamos de cargar, no con supuestos.

In [ ]:
dups = total - bronce_cc.drop("_fuente", "_archivo", "_fecha_ingesta").distinct().count()
fraudes = bronce_cc.filter("Class = 1").count()
monto_fraude = bronce_cc.filter("Class = 1").agg(F.sum("Amount")).collect()[0][0]
cinco_v = pd.DataFrame([
    {"V": "Volumen", "evidencia": f"F1: {meta_f1['bytes']/1024**2:.1f} MB y {total:,} registros · F2: {meta_f2['bytes']/1024**2:.1f} MB y {total_sfc:,} registros"},
    {"V": "Velocidad", "evidencia": "F1: ~142.000 transacciones/día (≈1,6 por segundo) en el periodo original · F2: actualización mensual de la SFC"},
    {"V": "Variedad", "evidencia": "2 fuentes, 2 formatos de origen (archivo CSV y API REST SODA); datos numéricos anonimizados (PCA) + datos categóricos y temporales"},
    {"V": "Veracidad", "evidencia": f"F1: {int((nulos_cc['nulos']>0).sum())} columnas con nulos, {dups:,} filas duplicadas ({dups/total*100:.2f} %) · F2: % nulos máx. por columna = {nulos_sfc['pct_nulos'].max():.2f} %"},
    {"V": "Valor", "evidencia": f"{fraudes:,} fraudes ({fraudes/total*100:.3f} %) por {monto_fraude:,.2f} EUR en 2 días; detectar el fraude sin bloquear clientes legítimos es el KPI del caso"},
])
cinco_v.to_csv(f"{EVID}/tablas/cinco_v.csv", index=False)
mostrar(cinco_v)

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)